# Minimum Viable Product
This will have a recommendation system with content-based filtering.
It will try to adjust the weights of the different features to see how it affects the recommendations.
e.g. directors matter more than actors


In [2]:
import json
from langchain.chains import LLMChain
import pprint


In [1]:
from langchain_ollama import OllamaLLM  # Local model

llm = OllamaLLM(model="deepseek-r1:1.5b")

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate

chain = (
    PromptTemplate.from_template(
        """Given the user question below, classify it as either being about `LangChain`, `Anthropic`, or `Other`.

Do not respond with more than one word.

<question>
{question}
</question>

Classification:"""
    )
    | OllamaLLM(model="deepseek-r1:1.5b")
    | StrOutputParser()
)

chain.invoke({"question": "how do I call Anthropic?"})

In [3]:
from langchain_core.prompts import PromptTemplate

sentiment_prompt = PromptTemplate.from_template("""
Analyze the sentiment of the following sentence.

Provide sentiment scores for only items within features like the following, and not the features themselves.
Don't provide sentiment scores for the features themselves, only the items within the features:
- Films
- Actors
- Directors
- Genres

Now, provide the sentiment scores for the features themselves, being the importance of the features to the user.

Provide a JSON response with sentiment scores (-1 to 1) for:
- Item ratings; Give the ratings of items within a feature, for example scores of various films.
- Feature importance; the importance of the features to the user. E.g. Importance of actors to the user.


Don't be verbose, just provide the scores with no other info.
Sentence: {input_text}
""")


In [4]:

sentiment_chain = LLMChain(
    llm=llm, 
    prompt=sentiment_prompt,
    output_key="sentiment_analysis"
)


/var/folders/lr/44zcpqfx2196g_qy5t3kc7m40000gp/T/ipykernel_64390/2846145021.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  sentiment_chain = LLMChain(


In [5]:
result = sentiment_chain.invoke(
    input="I like the film bullet train. In my opinion, directors are more important in films, I don't think actors are important.",
)

# print(result)
pprint.pprint(result)

{'input_text': 'I like the film bullet train. In my opinion, directors are '
               "more important in films, I don't think actors are important.",
 'sentiment_analysis': '<think>\n'
                       'Alright, let me try to figure out how to analyze the '
                       'sentiment of the given sentence and then determine the '
                       'feature importance for each category.\n'
                       '\n'
                       'First, looking at the sentence: "I like the film '
                       'bullet train. In my opinion, directors are more '
                       "important in films, I don't think actors are "
                       'important."\n'
                       '\n'
                       'I need to split this into items within the features. '
                       'The main features here are Films, Actors, Directors, '
                       "and Genres. So I'll analyze each of these aspects one "
                       'by one.

In [6]:
{'input_text': 'I like the film bullet train. In my opinion, directors are '
               "more important in films, I don't think actors are important. I "
               "don't mind what genre it is tbh. I do think Paul Mescal is a "
               "good actor though. I'm not sure where I sit on my opinion of "
               "the Hunger Games, I think it's a bit overrated. I do like the "
               "film The Dark Knight though, it's a classic. Also, I believe "
               'the music composer can make or break the film',
 'sentiment_analysis': 'For items within features:\n'
                       '- Films: 0.8 (bullet train), -0.5 (Hunger Games)\n'
                       '- Actors: 0.9 (Paul Mescal)\n'
                       '- Directors: 0.3 (bullet train, Hunger Games) \n'
                       '- Genres: 0 \n'
                       '\n'
                       'For feature importance:\n'
                       '- Films: 0.95\n'
                       '- Actors: 0.4\n'
                       '- Directors: 0.7\n'
                       '- Genres: -0.2'}

{'input_text': "I like the film bullet train. In my opinion, directors are more important in films, I don't think actors are important. I don't mind what genre it is tbh. I do think Paul Mescal is a good actor though. I'm not sure where I sit on my opinion of the Hunger Games, I think it's a bit overrated. I do like the film The Dark Knight though, it's a classic. Also, I believe the music composer can make or break the film",
 'sentiment_analysis': 'For items within features:\n- Films: 0.8 (bullet train), -0.5 (Hunger Games)\n- Actors: 0.9 (Paul Mescal)\n- Directors: 0.3 (bullet train, Hunger Games) \n- Genres: 0 \n\nFor feature importance:\n- Films: 0.95\n- Actors: 0.4\n- Directors: 0.7\n- Genres: -0.2'}

In [10]:


userprofile = {
    "genres": {"horror": -0.8, "action": 0.7, "comedy": 0.5, "romance": 0.3, "drama": 0.1, "suspense": 0.7, "mystery": 0.5, "science fiction": 0.6},
    "directors": {"Quentin Tarantino": 0.8, "Steven Spielberg": 0.7, "Martin Scorsese": 0.6, "Christopher Nolan": 0.5, "David Fincher": 0.4},
    "actors": {"Tom Hanks": 0.8, "Leonardo DiCaprio": 0.7, "Brad Pitt": 0.6, "Johnny Depp": 0.5, "Robert De Niro": 0.4},
    "metrics_weights_to_user": {"genres": 0.5, "directors": 0.4, "actors": 0.15}
    
    
}



In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import gradio as gr
from langchain.chains import LLMChain
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM  # Local model

# Initialize the model and tokenizer
checkpoint = "meta-llama/Llama-3.2-1B"
device = "cpu"  # "cuda" or "cpu"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)

# Define a chat template
chat_template = """
<|im_start|>{role}\n{content}
"""

# Define the predict function
def predict(message, history):
    history.append({"role": "user", "content": message})
    input_text = chat_template.format(role="user", content=message)
    inputs = tokenizer.encode(input_text, return_tensors="pt").to(device)
    outputs = model.generate(
        inputs,
        max_new_tokens=100,
        temperature=0.7,  # Increase temperature for more diverse responses
        top_p=0.9,       # Use nucleus sampling
        repetition_penalty=1.2,  # Penalize repetition
        do_sample=True
    )
    decoded = tokenizer.decode(outputs[0])
    response = decoded.split("<|im_start|>assistant\n")[-1].split("\n")[0]
    return response, history

# Create the Gradio interface
iface = gr.Interface(
    fn=predict,
    inputs=["text", "state"],
    outputs=["text", "state"],
    live=True
)

iface.launch()

Running on local URL:  http://127.0.0.1:7861

To create a public link, set `share=True` in `launch()`.


In [11]:
import gradio as gr
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate

# Initialize the OllamaLLM model
llm = OllamaLLM(model="llama3.2:3b-instruct-q4_K_M")

# Define a chat template
chat_template = """<|im_start|>{role}\n{content}\n"""

# Define the predict function
def predict(message, history):
    history.append({"role": "user", "content": message})
    input_text = chat_template.format(role="user", content=message)
    
    # Debug: Print input text
    print(f"Input Text: {input_text}")

    # Use the OllamaLLM model to generate a response
    try:
        response = llm.invoke({"prompt": input_text})
        # Debug: Print response
        print(f"Response: {response}")
    except Exception as e:
        # Debug: Print error
        print(f"Error: {e}")
        response = "An error occurred while generating the response."

    history.append({"role": "assistant", "content": response})
    return response, history

# Create the Gradio interface
iface = gr.Interface(
    fn=predict,
    inputs=["text", "state"],
    outputs=["text", "state"],
    live=True
)

iface.launch()

Running on local URL:  http://127.0.0.1:7864

To create a public link, set `share=True` in `launch()`.
